# NLP feature extraction for Bulgarian emotion recognition

Part-of-speech tagging, TF-IDF, sentiment, pretrained and custom word embeddings, and one extra feature, computed on the Bulgarian test-set sentences.

## Subtask 1: Part_of_speech (POS) tagging

In [ ]:
# Language Configuration - ONLY CHANGE THIS PARAMETER
LANGUAGE = 'bg'  # Change to 'en', 'de', 'fr', etc. for other languages

LANGUAGE_CONFIG = {
    'bg': {
        'spacy_model': 'bg_news_lg',
        'text_column': 'sentence_text_bg',
        'fasttext_model': 'cc.bg.300.bin',
        'corpus_file': 'bulgarian_corpus.txt',
        'sentiment_model': 'lxyuan/distilbert-base-multilingual-cased-sentiments-student'
    },
    'en': {
        'spacy_model': 'en_core_web_lg',
        'text_column': 'sentence_text',
        'fasttext_model': 'cc.en.300.bin',
        'corpus_file': 'english_corpus.txt',
        'sentiment_model': 'cardiffnlp/twitter-roberta-base-sentiment-latest'
    }
}

config = LANGUAGE_CONFIG[LANGUAGE]

In [1]:
import spacy
import pandas as pd
print(spacy.__version__)

3.5.4


In [2]:
import chardet

In [3]:
# Detect the encoding of the CSV file
with open("bg_final.csv", "rb") as file:
    result = chardet.detect(file.read())
    print(result)

{'encoding': 'UTF-8-SIG', 'confidence': 1.0, 'language': ''}


This was done since a few models were tried and their encoding was not UTF-8 and needed to be specified.

In [4]:
# Load the spaCy model for selected language
nlp = spacy.load(config['spacy_model'])
df = pd.read_csv("bg_final.csv", nrows=10)

In [5]:
df.head()

,dialogue_id,utterance_id,sentence_id,topic,dialog_act,emotion,sentence_text,sentence_text_bg
0,0,0,0,Ordinary Life,directive,disgust,The kitchen stinks .,Кухнята мирише.
1,0,1,0,Ordinary Life,commissive,no emotion,I'll throw out the garbage .,Ще изхвърля боклука.
2,1,0,0,Ordinary Life,directive,happiness,"So Dick , how about getting some coffee for to...","Така че Dick , какво ще кажете за получаване н..."
3,1,1,0,Ordinary Life,commissive,disgust,Coffee ?,Кафе ?
4,1,1,1,Ordinary Life,commissive,disgust,I don ’ t honestly like that kind of stuff .,"Честно казано, не харесвам такива неща."


In [6]:
# Process each sentence and store POS tags in a new column
df['POS_Tags'] = df['sentence_text_bg'].apply(lambda text: [token.pos_ for token in nlp(text)])

# Display the updated DataFrame
df.head(10)

,dialogue_id,utterance_id,sentence_id,topic,dialog_act,emotion,sentence_text,sentence_text_bg,POS_Tags
0,0,0,0,Ordinary Life,directive,disgust,The kitchen stinks .,Кухнята мирише.,"[NOUN, VERB, PUNCT]"
1,0,1,0,Ordinary Life,commissive,no emotion,I'll throw out the garbage .,Ще изхвърля боклука.,"[AUX, VERB, NOUN, PUNCT]"
2,1,0,0,Ordinary Life,directive,happiness,"So Dick , how about getting some coffee for to...","Така че Dick , какво ще кажете за получаване н...","[ADV, SCONJ, PROPN, PUNCT, DET, AUX, VERB, ADP..."
3,1,1,0,Ordinary Life,commissive,disgust,Coffee ?,Кафе ?,"[NOUN, PUNCT]"
4,1,1,1,Ordinary Life,commissive,disgust,I don ’ t honestly like that kind of stuff .,"Честно казано, не харесвам такива неща.","[ADV, ADJ, PUNCT, PART, VERB, DET, DET, PUNCT]"
5,1,2,0,Ordinary Life,directive,no emotion,"Come on , you can at least try a little , besi...","Хайде , можеш поне да опиташ малко , освен циг...","[PART, PUNCT, VERB, ADV, AUX, VERB, ADV, PUNCT..."
6,1,3,0,Ordinary Life,inform,anger,What ’ s wrong with that ?,Какво не е наред с това?,"[DET, PART, AUX, ADV, ADP, PRON, PUNCT]"
7,1,3,1,Ordinary Life,inform,anger,Cigarette is the thing I go crazy for .,"Цигарата е нещото, по което полудявам.","[NOUN, AUX, NOUN, PUNCT, ADP, PRON, VERB, PUNCT]"
8,1,4,0,Ordinary Life,inform,no emotion,"Not for me , Dick .","Не и за мен, Дик.","[PART, CCONJ, ADP, PRON, PUNCT, PROPN, PUNCT]"
9,2,0,0,Ordinary Life,question,no emotion,Are things still going badly with your housegu...,Все още ли нещата с вашия съквартирант вървят ...,"[ADV, ADV, PART, PRON, ADP, DET, NOUN, VERB, A..."


The bg_news_lg model is one of a small number of high-quality Bulgarian pipelines available for spaCy. It is a large model (version 3.5.4) trained on the Bulgarian Treebank (UD_Bulgarian-BTB) and includes a POS tagger component. The “large” size with ~200,000 Floret word vectors of dimension 300 improves its ability to cover diverse word forms, reducing the number of out-of-vocabulary tokens. Although it was trained on news domain text , which is more formal and grammatical , the model is is among the few that integrate directly into the spaCy pipeline, with large pretrained vectors, and is very easy to use for feature extraction.

## Subtask 2: Term Frequency-Inverse Document Frequency (TF-IDF)

In [7]:
# Import necessary libraries
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the text data to get the TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(df['sentence_text_bg'])

def get_nonzero_with_features(row):
    dense_row = row.toarray().flatten()
    nonzero_indices = dense_row.nonzero()[0]
    nonzero_values = dense_row[nonzero_indices]
    feature_names = vectorizer.get_feature_names_out()
    return [(feature_names[idx], float(val)) for idx, val in zip(nonzero_indices, nonzero_values)]

# Create a column "TF-IDF" in the DataFrame to store the TF-IDF vectors
df["TF-IDF"] = [get_nonzero_with_features(row) for row in tfidf_matrix]

df.head(10)

,dialogue_id,utterance_id,sentence_id,topic,dialog_act,emotion,sentence_text,sentence_text_bg,POS_Tags,TF-IDF
0,0,0,0,Ordinary Life,directive,disgust,The kitchen stinks .,Кухнята мирише.,"[NOUN, VERB, PUNCT]","[(кухнята, 0.7071067811865475), (мирише, 0.707..."
1,0,1,0,Ordinary Life,commissive,no emotion,I'll throw out the garbage .,Ще изхвърля боклука.,"[AUX, VERB, NOUN, PUNCT]","[(боклука, 0.6060433187339399), (изхвърля, 0.6..."
2,1,0,0,Ordinary Life,directive,happiness,"So Dick , how about getting some coffee for to...","Така че Dick , какво ще кажете за получаване н...","[ADV, SCONJ, PROPN, PUNCT, DET, AUX, VERB, ADP...","[(dick, 0.2693740845690302), (вечер, 0.2693740..."
3,1,1,0,Ordinary Life,commissive,disgust,Coffee ?,Кафе ?,"[NOUN, PUNCT]","[(кафе, 1.0)]"
4,1,1,1,Ordinary Life,commissive,disgust,I don ’ t honestly like that kind of stuff .,"Честно казано, не харесвам такива неща.","[ADV, ADJ, PUNCT, PART, VERB, DET, DET, PUNCT]","[(казано, 0.42435658024974), (не, 0.3156065018..."
5,1,2,0,Ordinary Life,directive,no emotion,"Come on , you can at least try a little , besi...","Хайде , можеш поне да опиташ малко , освен циг...","[PART, PUNCT, VERB, ADV, AUX, VERB, ADV, PUNCT...","[(да, 0.34410595277587486), (малко, 0.29252149..."
6,1,3,0,Ordinary Life,inform,anger,What ’ s wrong with that ?,Какво не е наред с това?,"[DET, PART, AUX, ADV, ADP, PRON, PUNCT]","[(какво, 0.46968603824569793), (наред, 0.55251..."
7,1,3,1,Ordinary Life,inform,anger,Cigarette is the thing I go crazy for .,"Цигарата е нещото, по което полудявам.","[NOUN, AUX, NOUN, PUNCT, ADP, PRON, VERB, PUNCT]","[(което, 0.4601578921271882), (нещото, 0.46015..."
8,1,4,0,Ordinary Life,inform,no emotion,"Not for me , Dick .","Не и за мен, Дик.","[PART, CCONJ, ADP, PRON, PUNCT, PROPN, PUNCT]","[(дик, 0.5525124243793842), (за, 0.46968603824..."
9,2,0,0,Ordinary Life,question,no emotion,Are things still going badly with your housegu...,Все още ли нещата с вашия съквартирант вървят ...,"[ADV, ADV, PART, PRON, ADP, DET, NOUN, VERB, A...","[(вашия, 0.35355339059327373), (все, 0.3535533..."


# Subtask 3: Sentiment Analysis

In [8]:
pip install torch torchvision torchaudio --index-url

Note: you may need to restart the kernel to use updated packages.


In [9]:
# Check PyTorch version
import torch
print(torch.__version__)

2.8.0+cpu


In [10]:
from transformers import pipeline

# Load a multilingual sentiment analysis model
sentiment_classifier = pipeline(
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    top_k=None
)

# Extract sentiment scores (positive - negative) for consistency with "Sentiment_Score"
def get_sentiment_score(text):
    result = sentiment_classifier(text)[0]  # list of dicts
    scores = {r['label']: r['score'] for r in result}
    return scores.get('positive', 0) - scores.get('negative', 0)

df['Sentiment_Score'] = df['sentence_text_bg'].apply(get_sentiment_score)
df.head(10)

,dialogue_id,utterance_id,sentence_id,topic,dialog_act,emotion,sentence_text,sentence_text_bg,POS_Tags,TF-IDF,Sentiment_Score
0,0,0,0,Ordinary Life,directive,disgust,The kitchen stinks .,Кухнята мирише.,"[NOUN, VERB, PUNCT]","[(кухнята, 0.7071067811865475), (мирише, 0.707...",0.385488
1,0,1,0,Ordinary Life,commissive,no emotion,I'll throw out the garbage .,Ще изхвърля боклука.,"[AUX, VERB, NOUN, PUNCT]","[(боклука, 0.6060433187339399), (изхвърля, 0.6...",-0.184246
2,1,0,0,Ordinary Life,directive,happiness,"So Dick , how about getting some coffee for to...","Така че Dick , какво ще кажете за получаване н...","[ADV, SCONJ, PROPN, PUNCT, DET, AUX, VERB, ADP...","[(dick, 0.2693740845690302), (вечер, 0.2693740...",-0.122162
3,1,1,0,Ordinary Life,commissive,disgust,Coffee ?,Кафе ?,"[NOUN, PUNCT]","[(кафе, 1.0)]",-0.163817
4,1,1,1,Ordinary Life,commissive,disgust,I don ’ t honestly like that kind of stuff .,"Честно казано, не харесвам такива неща.","[ADV, ADJ, PUNCT, PART, VERB, DET, DET, PUNCT]","[(казано, 0.42435658024974), (не, 0.3156065018...",-0.176561
5,1,2,0,Ordinary Life,directive,no emotion,"Come on , you can at least try a little , besi...","Хайде , можеш поне да опиташ малко , освен циг...","[PART, PUNCT, VERB, ADV, AUX, VERB, ADV, PUNCT...","[(да, 0.34410595277587486), (малко, 0.29252149...",0.024798
6,1,3,0,Ordinary Life,inform,anger,What ’ s wrong with that ?,Какво не е наред с това?,"[DET, PART, AUX, ADV, ADP, PRON, PUNCT]","[(какво, 0.46968603824569793), (наред, 0.55251...",-0.198530
7,1,3,1,Ordinary Life,inform,anger,Cigarette is the thing I go crazy for .,"Цигарата е нещото, по което полудявам.","[NOUN, AUX, NOUN, PUNCT, ADP, PRON, VERB, PUNCT]","[(което, 0.4601578921271882), (нещото, 0.46015...",-0.239203
8,1,4,0,Ordinary Life,inform,no emotion,"Not for me , Dick .","Не и за мен, Дик.","[PART, CCONJ, ADP, PRON, PUNCT, PROPN, PUNCT]","[(дик, 0.5525124243793842), (за, 0.46968603824...",-0.075781
9,2,0,0,Ordinary Life,question,no emotion,Are things still going badly with your housegu...,Все още ли нещата с вашия съквартирант вървят ...,"[ADV, ADV, PART, PRON, ADP, DET, NOUN, VERB, A...","[(вашия, 0.35355339059327373), (все, 0.3535533...",-0.287767


The “lxyuan/distilbert-base-multilingual-cased-sentiments-student” model was selected because it is a distilled multilingual model that supports Bulgarian among other languages. Given that TextBlob and VADER do not reliably cover Bulgarian sentiment, this model offers a more suitable alternative. It balances efficiency (being a student-distilled model) with multilingual capability, making it a practical choice for extracting sentiment scores for Bulgarian transcripts.

## Subtask 4: Pretrained Word Embeddings

In [11]:
import pandas as pd
import numpy as np
from gensim.models.fasttext import load_facebook_model
import re

### Pretrained Word Embeddings with fastText


In [13]:
# Load the pretrained FastText model
ft_model = load_facebook_model('cc.bg.300.bin')
print("Model loaded successfully.")

text_column = 'sentence_text_bg'


def sentence_to_vector(sentence, model):
    word_vectors_model = model.wv
    words = re.findall(r'\b\w+\b', str(sentence).lower())

    word_vectors = []
    for word in words:
        if word in word_vectors_model:
            word_vectors.append(word_vectors_model[word])

    if not word_vectors:
        # If no word matches, return a zero vector (same size as model’s vectors)
        return np.zeros(word_vectors_model.vector_size, dtype=np.float32)
    # Average the word vectors to get the sentence vector
    return np.mean(word_vectors, axis=0)


df['Pretrained_Embeddings'] = df[text_column].apply(lambda s: sentence_to_vector(s, ft_model))

print("\n--- Results ---")
print(df[['sentence_text_bg', 'Pretrained_Embeddings']].head())

Model loaded successfully.

--- Results ---
                                    sentence_text_bg  \
0                                    Кухнята мирише.   
1                               Ще изхвърля боклука.   
2  Така че Dick , какво ще кажете за получаване н...   
3                                             Кафе ?   
4            Честно казано, не харесвам такива неща.   

                               Pretrained_Embeddings  
0  [-0.05895353, 0.02678768, 0.011074896, 0.03241...  
1  [-0.05321903, 0.008844021, 0.07036609, 0.04865...  
2  [-0.0032325375, -0.014675511, 0.024256792, 0.0...  
3  [0.0019401921, 0.03712749, 0.025385574, 0.0153...  
4  [0.00017724124, -0.039979186, 0.014543567, 0.0...  


The Facebook fastText Filipino model “cc.bg.300.bin” was used because it provides pretrained word vectors for Bulgarian, trained on Common Crawl + Wikipedia data.
The vectors are 300-dimensional, and the model uses character n-grams (subword information) so that even rare or previously unseen word forms can be represented via their subword parts.
Because our transcripts are in Bulgarian, this model is well-matched in language, giving broad vocabulary coverage and allowing us to build sentence vectors by averaging. This method yields fixed-size embeddings for each sentence that can serve as input features.

More advanced models, such as FastText, address the OOV problem by generating word vectors from character n-grams (subwords).  For instance, the word apple can be broken down into subwords like <ap, app, ppl, ple>, and <apple>. If the word apples is not in the vocabulary, the model can still construct a vector for it by averaging the vectors of its known subwords. This is particularly effective for morphologically rich languages and for handling misspellings or rare words.


In [14]:
df.head(10)

,dialogue_id,utterance_id,sentence_id,topic,dialog_act,emotion,sentence_text,sentence_text_bg,POS_Tags,TF-IDF,Sentiment_Score,Pretrained_Embeddings
0,0,0,0,Ordinary Life,directive,disgust,The kitchen stinks .,Кухнята мирише.,"[NOUN, VERB, PUNCT]","[(кухнята, 0.7071067811865475), (мирише, 0.707...",0.385488,"[-0.05895353, 0.02678768, 0.011074896, 0.03241..."
1,0,1,0,Ordinary Life,commissive,no emotion,I'll throw out the garbage .,Ще изхвърля боклука.,"[AUX, VERB, NOUN, PUNCT]","[(боклука, 0.6060433187339399), (изхвърля, 0.6...",-0.184246,"[-0.05321903, 0.008844021, 0.07036609, 0.04865..."
2,1,0,0,Ordinary Life,directive,happiness,"So Dick , how about getting some coffee for to...","Така че Dick , какво ще кажете за получаване н...","[ADV, SCONJ, PROPN, PUNCT, DET, AUX, VERB, ADP...","[(dick, 0.2693740845690302), (вечер, 0.2693740...",-0.122162,"[-0.0032325375, -0.014675511, 0.024256792, 0.0..."
3,1,1,0,Ordinary Life,commissive,disgust,Coffee ?,Кафе ?,"[NOUN, PUNCT]","[(кафе, 1.0)]",-0.163817,"[0.0019401921, 0.03712749, 0.025385574, 0.0153..."
4,1,1,1,Ordinary Life,commissive,disgust,I don ’ t honestly like that kind of stuff .,"Честно казано, не харесвам такива неща.","[ADV, ADJ, PUNCT, PART, VERB, DET, DET, PUNCT]","[(казано, 0.42435658024974), (не, 0.3156065018...",-0.176561,"[0.00017724124, -0.039979186, 0.014543567, 0.0..."
5,1,2,0,Ordinary Life,directive,no emotion,"Come on , you can at least try a little , besi...","Хайде , можеш поне да опиташ малко , освен циг...","[PART, PUNCT, VERB, ADV, AUX, VERB, ADV, PUNCT...","[(да, 0.34410595277587486), (малко, 0.29252149...",0.024798,"[0.021950508, -0.041999117, 0.0140455365, 0.01..."
6,1,3,0,Ordinary Life,inform,anger,What ’ s wrong with that ?,Какво не е наред с това?,"[DET, PART, AUX, ADV, ADP, PRON, PUNCT]","[(какво, 0.46968603824569793), (наред, 0.55251...",-0.198530,"[-0.00087265484, -0.02994658, 0.005080737, 0.0..."
7,1,3,1,Ordinary Life,inform,anger,Cigarette is the thing I go crazy for .,"Цигарата е нещото, по което полудявам.","[NOUN, AUX, NOUN, PUNCT, ADP, PRON, VERB, PUNCT]","[(което, 0.4601578921271882), (нещото, 0.46015...",-0.239203,"[0.022024015, -0.015521896, -0.003359329, 0.00..."
8,1,4,0,Ordinary Life,inform,no emotion,"Not for me , Dick .","Не и за мен, Дик.","[PART, CCONJ, ADP, PRON, PUNCT, PROPN, PUNCT]","[(дик, 0.5525124243793842), (за, 0.46968603824...",-0.075781,"[-0.018605703, -0.014853599, -0.021822492, 0.0..."
9,2,0,0,Ordinary Life,question,no emotion,Are things still going badly with your housegu...,Все още ли нещата с вашия съквартирант вървят ...,"[ADV, ADV, PART, PRON, ADP, DET, NOUN, VERB, A...","[(вашия, 0.35355339059327373), (все, 0.3535533...",-0.287767,"[-0.018371671, -0.009949706, -0.030019002, 0.0..."


## Subtask 5: Custom Word Embedding Model

In [15]:
import logging
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

# Configure logging to monitor training progress
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

# Path to the text corpus file
corpus_file = 'bulgarian_corpus.txt'
print(f"Loading sentences from {corpus_file}...")

# LineSentence streams the corpus sentence by sentence to save memory
sentences = LineSentence(corpus_file)

# Train a Word2Vec model on the corpus
custom_model = Word2Vec(
    sentences=sentences,
    vector_size=150,
    window=5,
    min_count=5,
    sg=1,
    workers=4
)

# Save the trained model to disk
model_save_path = "bulgarian_wiki_word2vec.model"
custom_model.save(model_save_path)

# Column to store custom embeddings
df['Custom_Embeddings'] = df[text_column].apply(lambda s: sentence_to_vector(s, custom_model))

df.head(10)

2025-09-18 15:20:22,913 : INFO : collecting all words and their counts
2025-09-18 15:20:22,917 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types


Loading sentences from bulgarian_corpus.txt...


2025-09-18 15:20:24,444 : INFO : PROGRESS: at sentence #10000, processed 6963634 words, keeping 309368 word types
2025-09-18 15:20:25,900 : INFO : PROGRESS: at sentence #20000, processed 13561591 words, keeping 483464 word types
2025-09-18 15:20:27,017 : INFO : PROGRESS: at sentence #30000, processed 18618160 words, keeping 595302 word types
2025-09-18 15:20:27,990 : INFO : PROGRESS: at sentence #40000, processed 23001334 words, keeping 678393 word types
2025-09-18 15:20:28,899 : INFO : PROGRESS: at sentence #50000, processed 26985870 words, keeping 760506 word types
2025-09-18 15:20:29,687 : INFO : PROGRESS: at sentence #60000, processed 30503578 words, keeping 830242 word types
2025-09-18 15:20:30,412 : INFO : PROGRESS: at sentence #70000, processed 33744747 words, keeping 886142 word types
2025-09-18 15:20:31,077 : INFO : PROGRESS: at sentence #80000, processed 36701749 words, keeping 932878 word types
2025-09-18 15:20:31,723 : INFO : PROGRESS: at sentence #90000, processed 39558483

,dialogue_id,utterance_id,sentence_id,topic,dialog_act,emotion,sentence_text,sentence_text_bg,POS_Tags,TF-IDF,Sentiment_Score,Pretrained_Embeddings,Custom_Embeddings
0,0,0,0,Ordinary Life,directive,disgust,The kitchen stinks .,Кухнята мирише.,"[NOUN, VERB, PUNCT]","[(кухнята, 0.7071067811865475), (мирише, 0.707...",0.385488,"[-0.05895353, 0.02678768, 0.011074896, 0.03241...","[0.45568153, -0.027227957, 0.09161861, -0.1835..."
1,0,1,0,Ordinary Life,commissive,no emotion,I'll throw out the garbage .,Ще изхвърля боклука.,"[AUX, VERB, NOUN, PUNCT]","[(боклука, 0.6060433187339399), (изхвърля, 0.6...",-0.184246,"[-0.05321903, 0.008844021, 0.07036609, 0.04865...","[0.2159934, -0.034524005, 0.06626176, 0.208314..."
2,1,0,0,Ordinary Life,directive,happiness,"So Dick , how about getting some coffee for to...","Така че Dick , какво ще кажете за получаване н...","[ADV, SCONJ, PROPN, PUNCT, DET, AUX, VERB, ADP...","[(dick, 0.2693740845690302), (вечер, 0.2693740...",-0.122162,"[-0.0032325375, -0.014675511, 0.024256792, 0.0...","[0.47156695, -0.14618982, -0.044646528, -0.045..."
3,1,1,0,Ordinary Life,commissive,disgust,Coffee ?,Кафе ?,"[NOUN, PUNCT]","[(кафе, 1.0)]",-0.163817,"[0.0019401921, 0.03712749, 0.025385574, 0.0153...","[0.6692529, -0.5150005, 0.54632986, -0.2239101..."
4,1,1,1,Ordinary Life,commissive,disgust,I don ’ t honestly like that kind of stuff .,"Честно казано, не харесвам такива неща.","[ADV, ADJ, PUNCT, PART, VERB, DET, DET, PUNCT]","[(казано, 0.42435658024974), (не, 0.3156065018...",-0.176561,"[0.00017724124, -0.039979186, 0.014543567, 0.0...","[0.5595237, -0.0887767, -0.089905106, 0.080638..."
5,1,2,0,Ordinary Life,directive,no emotion,"Come on , you can at least try a little , besi...","Хайде , можеш поне да опиташ малко , освен циг...","[PART, PUNCT, VERB, ADV, AUX, VERB, ADV, PUNCT...","[(да, 0.34410595277587486), (малко, 0.29252149...",0.024798,"[0.021950508, -0.041999117, 0.0140455365, 0.01...","[0.46075618, -0.087264046, -0.12859842, -0.088..."
6,1,3,0,Ordinary Life,inform,anger,What ’ s wrong with that ?,Какво не е наред с това?,"[DET, PART, AUX, ADV, ADP, PRON, PUNCT]","[(какво, 0.46968603824569793), (наред, 0.55251...",-0.198530,"[-0.00087265484, -0.02994658, 0.005080737, 0.0...","[0.4478457, -0.19270393, -0.08149754, -0.12825..."
7,1,3,1,Ordinary Life,inform,anger,Cigarette is the thing I go crazy for .,"Цигарата е нещото, по което полудявам.","[NOUN, AUX, NOUN, PUNCT, ADP, PRON, VERB, PUNCT]","[(което, 0.4601578921271882), (нещото, 0.46015...",-0.239203,"[0.022024015, -0.015521896, -0.003359329, 0.00...","[0.57746446, -0.017069545, 0.03995408, -0.0177..."
8,1,4,0,Ordinary Life,inform,no emotion,"Not for me , Dick .","Не и за мен, Дик.","[PART, CCONJ, ADP, PRON, PUNCT, PROPN, PUNCT]","[(дик, 0.5525124243793842), (за, 0.46968603824...",-0.075781,"[-0.018605703, -0.014853599, -0.021822492, 0.0...","[0.54453075, -0.071342036, 0.0049987193, 0.049..."
9,2,0,0,Ordinary Life,question,no emotion,Are things still going badly with your housegu...,Все още ли нещата с вашия съквартирант вървят ...,"[ADV, ADV, PART, PRON, ADP, DET, NOUN, VERB, A...","[(вашия, 0.35355339059327373), (все, 0.3535533...",-0.287767,"[-0.018371671, -0.009949706, -0.030019002, 0.0...","[0.24767524, -0.0904341, 0.043866098, 0.049756..."


In [ ]:
import logging
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

# --- Configuration ---
# Set up logging to monitor the training process of the Word2Vec models.
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

# --- Function Definition ---
def evaluate_word2vec_model(model, evaluation_dict, topn=30):
    """
    Evaluates the semantic similarity performance of a trained Word2Vec model.

    The function assesses the model by checking if a predefined list of semantically
    similar words for a given headword is present in the top 'n' most similar
    words predicted by the model.

    Args:
        model (gensim.models.Word2Vec): The trained Word2Vec model instance.
        evaluation_dict (dict): A dictionary where each key is a headword and its
                                value is a list of words expected to be similar.
        topn (int): The number of most similar words to retrieve from the model
                    for comparison.

    Returns:
        int: The total score, calculated as the cumulative number of successful
             matches found across all headwords in the evaluation dictionary.
    """
    total_score = 0
    print("\n--- Starting Model Evaluation ---")

    for head_word, expected_similar_words in evaluation_dict.items():
        # Verify that the headword exists in the model's vocabulary.
        if head_word not in model.wv:
            print(f"Warning: Headword '{head_word}' not found in model vocabulary. Skipping.")
            continue

        try:
            # Retrieve the top N most similar words predicted by the model.
            predicted_similar_words = [word for word, score in model.wv.most_similar(head_word, topn=topn)]

            # Identify the intersection between the expected and predicted lists.
            matches = set(expected_similar_words) & set(predicted_similar_words)

            # Calculate the score for the current headword.
            score = len(matches)
            total_score += score

            print(f"Headword '{head_word}': Found {score}/{len(expected_similar_words)} matches. "
                  f"Examples: {list(matches)[:5]}")

        except Exception as e:
            print(f"An error occurred while processing '{head_word}': {e}")

    print("--- Evaluation Complete ---")
    return total_score

# --- Main Execution Block ---

# 1. Corpus and Data Preparation
# Specify the path to the pre-processed text corpus.
# The corpus should be a single text file where each line constitutes a sentence.
corpus_file = 'bulgarian_corpus.txt'
sentences = LineSentence(corpus_file)

# 2. Hyperparameter Configuration
# Define a list of hyperparameter combinations to train and evaluate.
# 'sg=1' corresponds to the Skip-gram algorithm.
# 'sg=0' corresponds to the Continuous Bag-of-Words (CBOW) algorithm.
param_combinations = [
    # Skip-gram Models
    {'vector_size': 150, 'window': 5, 'sg': 1, 'min_count': 5},
    {'vector_size': 300, 'window': 5, 'sg': 1, 'min_count': 5},
    {'vector_size': 300, 'window': 10, 'sg': 1, 'min_count': 5},
    {'vector_size': 300, 'window': 5, 'sg': 1, 'min_count': 10},
    # CBOW Models
    {'vector_size': 150, 'window': 5, 'sg': 0, 'min_count': 5},
    {'vector_size': 300, 'window': 10, 'sg': 0, 'min_count': 5},
]

# 3. Evaluation Set Definition
# A custom dictionary to test the model's understanding of semantic similarity
# for a curated set of Bulgarian words.
bulgarian_test_words = {
    # Nouns
    'цар': ['крал', 'император', 'монарх', 'владетел', 'принц', 'престолонаследник', 'княз'],
    'град': ['столица', 'селище', 'квартал', 'местност', 'градче', 'мегаполис', 'община', 'населено_място'],
    'жена': ['съпруга', 'дама', 'майка', 'момиче', 'госпожа', 'млада_жена', 'девойка', 'матрона'],
    'кола': ['автомобил', 'превозно_средство', 'машина', 'камион', 'бус', 'колесар', 'мотор'],
    'учител': ['преподавател', 'педагог', 'наставник', 'ментор', 'образовател', 'лектор', 'треньор'],
    'ученик': ['студент', 'школник', 'учещ', 'курсист', 'стажант', 'абитуриент', 'възпитаник'],
    'книга': ['роман', 'повест', 'разказ', 'том', 'литература', 'четиво', 'издание', 'електронна_книга'],
    'информация': ['данни', 'сведения', 'факти', 'знание', 'известия', 'новини', 'съобщения', 'справка', 'справки'],
    'пътуване': ['обиколка', 'екскурзия', 'турне', 'пътешествие', 'посещение', 'почивка', 'обиколен_маршрут', 'транзит'],
    'свобода': ['независимост', 'автономия', 'воля', 'непринуденост', 'либерализъм', 'освобождение', 'революция'],
    'щастие': ['радост', 'веселие', 'доволство', 'благополучие', 'блаженство', 'удоволствие', 'наслада'],
    'природа': ['околна_среда', 'естество', 'флора', 'фауна', 'пейзаж', 'ландшафт', 'жива_природа'],
    # Adjectives
    'красив': ['хубав', 'прекрасен', 'очарователен', 'симпатичен', 'привлекателен', 'ослепителен', 'изящен', 'елегантен'],
    'голям': ['огромен', 'масивен', 'обемист', 'внушителен', 'колосален', 'гигантски', 'монументален', 'широк'],
    'малък': ['дребен', 'миниатюрен', 'ситен', 'компактен', 'нищожен', 'незначителен', 'кратък'],
    'добър': ['мил', 'почтен', 'честен', 'справедлив', 'качествен', 'приятен', 'успешен', 'достоен', 'отличен'],
    'тъжен': ['нещастен', 'печален', 'мрачен', 'депресиран', 'угнетен', 'отчаян', 'потиснат', 'самотен'],
    # Verbs
    'говоря': ['казвам', 'разговарям', 'приказвам', 'беседвам', 'шепна', 'рецитирам', 'изказвам_се', 'разказвам'],
    'правя': ['извършвам', 'създавам', 'изработвам', 'осъществявам', 'изпълнявам', 'построявам'],
    'отивам': ['тръгвам', 'заминавам', 'пътувам', 'ходя', 'пристигам', 'отида', 'придвижвам_се', 'отправям_се'],
    'мисля': ['размишлявам', 'разсъждавам', 'обмислям', 'умувам', 'преценявам', 'преосмислям', 'смятам', 'считам'],
}

# 4. Model Training and Selection Loop
best_score = -1
best_params = None
best_model = None

for i, params in enumerate(param_combinations, 1):
    print(f"\n--- Training Model {i}/{len(param_combinations)} ---")
    print(f"Parameters: {params}")

    # Initialize and train the Word2Vec model.
    model = Word2Vec(
        sentences=sentences,
        vector_size=params['vector_size'],
        window=params['window'],
        min_count=params['min_count'],
        sg=params['sg'],
        workers=4  # Specify the number of CPU cores for parallelization.
    )

    # Evaluate the newly trained model.
    current_score = evaluate_word2vec_model(model, bulgarian_test_words, topn=30)
    print(f"Result: Model {i} achieved a score of {current_score}.")

    # Update the best model if the current one performs better.
    if current_score > best_score:
        best_score = current_score
        best_params = params
        best_model = model
        print(f"Update: New best model identified with score {best_score}.")

# 5. Save the Best Performing Model
if best_model:
    model_save_path = "best_bulgarian_word2vec.model"
    best_model.save(model_save_path)
    print("\n--- Final Result ---")
    print(f"The best performing model achieved a score of {best_score}.")
    print(f"Optimal parameters: {best_params}")
    print(f"Model saved to file: '{model_save_path}'")
else:
    print("\nTraining process concluded, but no models were successfully trained or evaluated.")

The best model achieved a score of 41 with parameters: {'vector_size': 150, 'window': 5, 'sg': 0, 'min_count': 5}

sg:0
It was more efficient at the end, since its faster and requires less data. Assumptions were made that we have a big enough dataset which was not the case. It learns to predict a word from its surrounding context. It works better on a smaller dataset.

Vector_size:150
This choice balances computational efficiency with model capacity. Lower-dimensional vectors (150) require less memory and training time compared to higher dimensions like 300. 300+ leads to overfitting due to the size of the dataset. The smaller vector size better captures the features presented in the data.

Min_count: 5
It is a standard choice in the industry since min_count=5 filters out very rare words that would have poor vector representations due to insufficient training examples, while preserving enough vocabulary for good coverage.

Epochs: The gensim library, by default, trains for 5 epochs. 1 epoch is mostly never enough. It could be compared to reading a book once. On the second you look for connections and usually on the 4/5th time you can make a proper analysis. More epochs might lead to overfitting

### Testing and Comparing Both Models

In [16]:
from gensim.models import Word2Vec

model = Word2Vec.load("bulgarian_wiki_word2vec.model")

# Find the 10 most similar words to "България" (Bulgaria)
try:
    similar_words = model.wv.most_similar('българия', topn=10)
    print("Words similar to 'България':")
    for word, score in similar_words:
        print(f"- {word}: {score:.4f}")
except KeyError:
    print("The word 'българия' is not in the model's vocabulary. Try another common word.")

# Find the 10 most similar words to "цар" (tsar/king)
try:
    similar_words = model.wv.most_similar('цар', topn=10)
    print("\nWords similar to 'цар':")
    for word, score in similar_words:
        print(f"- {word}: {score:.4f}")
except KeyError:
    print("The word 'цар' is not in the model's vocabulary. Try another common word.")

2025-09-18 15:38:10,294 : INFO : loading Word2Vec object from bulgarian_wiki_word2vec.model
2025-09-18 15:38:10,559 : INFO : loading wv recursively from bulgarian_wiki_word2vec.model.wv.* with mmap=None
2025-09-18 15:38:10,560 : INFO : loading vectors from bulgarian_wiki_word2vec.model.wv.vectors.npy with mmap=None
2025-09-18 15:38:10,614 : INFO : loading syn1neg from bulgarian_wiki_word2vec.model.syn1neg.npy with mmap=None
2025-09-18 15:38:10,671 : INFO : setting ignored attribute cum_table to None
2025-09-18 15:38:12,315 : INFO : Word2Vec lifecycle event {'fname': 'bulgarian_wiki_word2vec.model', 'datetime': '2025-09-18T15:38:12.314440', 'gensim': '4.3.3', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26100-SP0', 'event': 'loaded'}


Words similar to 'България':
- асеновград: 0.5866
- бълария: 0.5834
- софия: 0.5771
- горянско: 0.5721
- русофилство: 0.5709
- пловдив: 0.5703
- потв: 0.5646
- кюстендил: 0.5585
- нуртс: 0.5582
- утрешна: 0.5576

Words similar to 'цар':
- царица: 0.7548
- царят: 0.7548
- царя: 0.7471
- царете: 0.7053
- касандър: 0.6790
- обединител: 0.6733
- сенекерим: 0.6711
- лидийския: 0.6711
- хасмонейския: 0.6650
- гигес: 0.6531


In [18]:
from gensim.models import KeyedVectors, Word2Vec

# Load both models
ft_model = load_facebook_model('cc.bg.300.bin')

custom_model_path = "bulgarian_wiki_word2vec.model"

custom_model = Word2Vec.load(custom_model_path).wv


# Define Words to Test
words_to_test = ['жена', 'пич', 'полицай']

# Comparison
for word in words_to_test:
    print(f"\n--- Comparing results for the word: '{word}' ---")

    try:
        print("\nPretrained FastText Results:")
        ft_similar = ft_model.wv.most_similar(word, topn=7)
        for term, score in ft_similar:
            print(f"- {term}: {score:.4f}")
    except KeyError:
        print(f"'{word}' not found in the FastText model.")

    try:
        print("\nCustom Word2Vec Results:")
        custom_similar = custom_model.most_similar(word, topn=7)
        for term, score in custom_similar:
            print(f"- {term}: {score:.4f}")
    except KeyError:
        print(f"'{word}' not found in the custom model.")

2025-09-18 15:46:55,906 : INFO : loading 2000000 words for fastText model from cc.bg.300.bin


MemoryError: Unable to allocate 4.47 GiB for an array with shape (1200000000,) and data type float32

In [19]:
import pandas as pd
import numpy as np
from gensim.models import KeyedVectors, Word2Vec
import re
import pickle

# Load FastText model
ft_model = load_facebook_model('cc.bg.300.bin')

# Load Custom Word2Vec model
custom_model = Word2Vec.load("bulgarian_wiki_word2vec.model")
print("Custom model loaded.")

df = pd.read_csv("bg_final.csv")
text_column = 'sentence_text_bg'
print(f"Dataset 'bg_final.csv' loaded with {len(df)} rows.")

# Function to convert sentences to vectors
def sentence_to_vector(sentence, model):
    """
    Converts a sentence string into a vector by averaging its word vectors.
    Returns a zero vector if no words are found in the model's vocabulary.
    """
    words = re.findall(r'\b\w+\b', str(sentence).lower())

    word_vectors_source = model.wv if hasattr(model, 'wv') else model

    vector_size = word_vectors_source.vector_size
    word_vectors = []

    for word in words:
        if word in word_vectors_source:
            word_vectors.append(word_vectors_source[word])

    if not word_vectors:
        return np.zeros(vector_size, dtype=np.float32)

    return np.mean(word_vectors, axis=0)

# Generate sentence vectors for the FastText model
df['sentence_vector_fasttext'] = df[text_column].apply(
    lambda s: sentence_to_vector(s, ft_model)
)

# Generate sentence vectors for the Custom model
df['sentence_vector_custom'] = df[text_column].apply(
    lambda s: sentence_to_vector(s, custom_model)
)
print("Custom model vectors created.")

# Save the DataFrame with new features to a pickle file
output_path = 'bg_features.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(df, f)

print(f"Your new DataFrame with vector features has been saved to: {output_path}")

print("\nSample of the final DataFrame:")
print(df[['sentence_text_bg', 'sentence_vector_custom', 'sentence_vector_fasttext']].head())

2025-09-18 15:47:45,071 : INFO : loading 2000000 words for fastText model from cc.bg.300.bin


MemoryError: Unable to allocate 4.47 GiB for an array with shape (1200000000,) and data type float32

## Subtask 6: Additional Feature

In [20]:
# 1. Word Count: The total number of words in the sentence.
df['word_count'] = df['sentence_text_bg'].apply(lambda x: len(str(x).split()))

# 2. Exclamation Mark Count: A strong signal for high-arousal emotions.
df['exclamation_marks'] = df['sentence_text_bg'].str.count('!')

# 3. Question Mark Count: Can signal surprise, confusion, or anger.
df['question_marks'] = df['sentence_text_bg'].str.count('\?')

# 4. Uppercase Word Count: A proxy for "shouting" in text.
df['uppercase_words'] = df['sentence_text_bg'].apply(
    lambda x: sum(1 for word in str(x).split() if word.isupper() and len(word) > 1)
)
df.head(10)

,dialogue_id,utterance_id,sentence_id,topic,dialog_act,emotion,sentence_text,sentence_text_bg,POS_Tags,TF-IDF,Sentiment_Score,Pretrained_Embeddings,Custom_Embeddings,word_count,exclamation_marks,question_marks,uppercase_words
0,0,0,0,Ordinary Life,directive,disgust,The kitchen stinks .,Кухнята мирише.,"[NOUN, VERB, PUNCT]","[(кухнята, 0.7071067811865475), (мирише, 0.707...",0.385488,"[-0.05895353, 0.02678768, 0.011074896, 0.03241...","[0.45568153, -0.027227957, 0.09161861, -0.1835...",2,0,0,0
1,0,1,0,Ordinary Life,commissive,no emotion,I'll throw out the garbage .,Ще изхвърля боклука.,"[AUX, VERB, NOUN, PUNCT]","[(боклука, 0.6060433187339399), (изхвърля, 0.6...",-0.184246,"[-0.05321903, 0.008844021, 0.07036609, 0.04865...","[0.2159934, -0.034524005, 0.06626176, 0.208314...",3,0,0,0
2,1,0,0,Ordinary Life,directive,happiness,"So Dick , how about getting some coffee for to...","Така че Dick , какво ще кажете за получаване н...","[ADV, SCONJ, PROPN, PUNCT, DET, AUX, VERB, ADP...","[(dick, 0.2693740845690302), (вечер, 0.2693740...",-0.122162,"[-0.0032325375, -0.014675511, 0.024256792, 0.0...","[0.47156695, -0.14618982, -0.044646528, -0.045...",15,0,1,0
3,1,1,0,Ordinary Life,commissive,disgust,Coffee ?,Кафе ?,"[NOUN, PUNCT]","[(кафе, 1.0)]",-0.163817,"[0.0019401921, 0.03712749, 0.025385574, 0.0153...","[0.6692529, -0.5150005, 0.54632986, -0.2239101...",2,0,1,0
4,1,1,1,Ordinary Life,commissive,disgust,I don ’ t honestly like that kind of stuff .,"Честно казано, не харесвам такива неща.","[ADV, ADJ, PUNCT, PART, VERB, DET, DET, PUNCT]","[(казано, 0.42435658024974), (не, 0.3156065018...",-0.176561,"[0.00017724124, -0.039979186, 0.014543567, 0.0...","[0.5595237, -0.0887767, -0.089905106, 0.080638...",6,0,0,0
5,1,2,0,Ordinary Life,directive,no emotion,"Come on , you can at least try a little , besi...","Хайде , можеш поне да опиташ малко , освен циг...","[PART, PUNCT, VERB, ADV, AUX, VERB, ADV, PUNCT...","[(да, 0.34410595277587486), (малко, 0.29252149...",0.024798,"[0.021950508, -0.041999117, 0.0140455365, 0.01...","[0.46075618, -0.087264046, -0.12859842, -0.088...",12,0,0,0
6,1,3,0,Ordinary Life,inform,anger,What ’ s wrong with that ?,Какво не е наред с това?,"[DET, PART, AUX, ADV, ADP, PRON, PUNCT]","[(какво, 0.46968603824569793), (наред, 0.55251...",-0.198530,"[-0.00087265484, -0.02994658, 0.005080737, 0.0...","[0.4478457, -0.19270393, -0.08149754, -0.12825...",6,0,1,0
7,1,3,1,Ordinary Life,inform,anger,Cigarette is the thing I go crazy for .,"Цигарата е нещото, по което полудявам.","[NOUN, AUX, NOUN, PUNCT, ADP, PRON, VERB, PUNCT]","[(което, 0.4601578921271882), (нещото, 0.46015...",-0.239203,"[0.022024015, -0.015521896, -0.003359329, 0.00...","[0.57746446, -0.017069545, 0.03995408, -0.0177...",6,0,0,0
8,1,4,0,Ordinary Life,inform,no emotion,"Not for me , Dick .","Не и за мен, Дик.","[PART, CCONJ, ADP, PRON, PUNCT, PROPN, PUNCT]","[(дик, 0.5525124243793842), (за, 0.46968603824...",-0.075781,"[-0.018605703, -0.014853599, -0.021822492, 0.0...","[0.54453075, -0.071342036, 0.0049987193, 0.049...",5,0,0,0
9,2,0,0,Ordinary Life,question,no emotion,Are things still going badly with your housegu...,Все още ли нещата с вашия съквартирант вървят ...,"[ADV, ADV, PART, PRON, ADP, DET, NOUN, VERB, A...","[(вашия, 0.35355339059327373), (все, 0.3535533...",-0.287767,"[-0.018371671, -0.009949706, -0.030019002, 0.0...","[0.24767524, -0.0904341, 0.043866098, 0.049756...",9,0,1,0


In [22]:
# Keep only the required columns
final_features = df[['sentence_text', 'sentence_text_bg', 'POS_Tags', 'TF-IDF', 'Sentiment_Score', 'Pretrained_Embeddings', 'Custom_Embeddings', 'word_count', 'exclamation_marks', 'question_marks', 'uppercase_words']]

# Save the first 10 rows as required deliverable
final_features.head(10).to_csv("NLP_features.csv", sep="\t", index=False)